In [7]:
import json
import pandas as pd
import networkx as nx
from pathlib import Path
from collections import Counter

RESULTS_DIR = Path("../data/results")
TRIPLES_FILE    = RESULTS_DIR / "community_triples.json"
GRAPH_FILE      = RESULTS_DIR / "knowledge_graph.graphml"
NODE_FILE       = RESULTS_DIR / "knowledge_graph_nodes.csv"
EDGE_FILE       = RESULTS_DIR / "knowledge_graph_edges.csv"
METRICS_FILE    = RESULTS_DIR / "knowledge_graph_metrics.json"

with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

print(f"Loaded triples from {len(community_triples)} communities")
print("Sample:", list(community_triples.items())[0])

Loaded triples from 19 communities
Sample: ('0', [{'subject': 'ftp_client', 'relation': 'targets', 'target': 'ftp_port_21_service'}, {'subject': 'ftp_client', 'relation': 'scans', 'target': 'http_web_server'}, {'subject': 'http_flood_source', 'relation': 'floods', 'target': 'http_web_server'}, {'subject': 'http_flood_source', 'relation': 'targets', 'target': 'http_port_80_service'}])


In [8]:
# construct graph with edge weights based on triple frequency across communities
G = nx.DiGraph()
edge_weights = {} 
total_triples = 0
invalid_triples = 0

# track which communities mention each edge for later analysis
edge_communities = {}

for cid, triples in community_triples.items():
    for t in triples:
        s = str(t.get("subject", "")).strip()
        r = str(t.get("relation", "")).strip()
        o = str(t.get("target", "")).strip()

        if not s or not r or not o:
            invalid_triples += 1
            continue

        total_triples += 1
        key = (s, r, o)
        edge_weights[key] = edge_weights.get(key, 0) + 1
        edge_communities.setdefault(key, []).append(cid)

for (src, rel, tgt), weight in edge_weights.items():
    G.add_node(src)
    G.add_node(tgt)
    G.add_edge(
        src, tgt,
        relation=rel,
        weight=weight,
        communities=",".join(edge_communities[(src, rel, tgt)])
    )

print(f"Graph constructed: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Processed {total_triples} triples | Skipped {invalid_triples} invalid")

Graph constructed: 38 nodes, 43 edges
Processed 76 triples | Skipped 0 invalid


In [9]:
# Analyze graph
print("TOP 10 NODES BY DEGREE")
for node, deg in sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {node} (degree={deg})")

print("\nTOP 10 EDGES BY WEIGHT")
edge_data = [(u, v, d['relation'], d['weight']) for u, v, d in G.edges(data=True)]
for u, v, rel, w in sorted(edge_data, key=lambda x: x[3], reverse=True)[:10]:
    print(f"  (weight={w}) {u} --[{rel}]--> {v}")

TOP 10 NODES BY DEGREE
  ftp_client (degree=10)
  http_web_server (degree=7)
  dns_resolver (degree=7)
  http_flood_source (degree=5)
  ssh_port_22_service (degree=5)
  high_volume_short_connections (degree=5)
  ssh_brute_force_client (degree=3)
  dns_client (degree=3)
  ssh_authentication_endpoint (degree=3)
  https_client (degree=3)

TOP 10 EDGES BY WEIGHT
  (weight=3) dns_resolver --[scans]--> http_web_server
  (weight=2) ftp_client --[scans]--> ftp_port_21_service
  (weight=2) ftp_client --[floods]--> network_bandwidth
  (weight=2) ssh_brute_force_client --[scans]--> ssh_port_22_service
  (weight=2) credential_guessing_process --[executes]--> password_spray_attack
  (weight=2) dns_client --[scans]--> dns_resolver
  (weight=2) attacker --[authenticates_to]--> ssh_authentication_endpoint
  (weight=1) ftp_client --[scans]--> http_web_server
  (weight=1) ftp_client --[targets]--> ftp_server
  (weight=1) ftp_client --[executes]--> ftp_operation


In [10]:
# Export graph structure
nx.write_graphml(G, GRAPH_FILE)

nodes_out = [{"node": n, "degree": d} for n, d in G.degree()]
pd.DataFrame(nodes_out).sort_values("degree", ascending=False).to_csv(NODE_FILE, index=False)

edges_out = [
    {"source": u, "target": v, "relation": data["relation"],
     "weight": data["weight"], "communities": data.get("communities", "")}
    for u, v, data in G.edges(data=True)
]
pd.DataFrame(edges_out).sort_values("weight", ascending=False).to_csv(EDGE_FILE, index=False)

In [11]:
# Compute and save graph metrics
relation_dist = Counter([data["relation"] for _, _, data in G.edges(data=True)])

metrics = {
    "construction_method": "LLM open-extraction (subject, relation, target) triples -> NetworkX DiGraph",
    "extraction_model": "qwen2.5-8b-instruct-q4_k_m",
    "total_triples_processed": total_triples,
    "invalid_triples_skipped": invalid_triples,
    "unique_nodes": G.number_of_nodes(),
    "unique_edges": G.number_of_edges(),
    "average_degree": round(sum(dict(G.degree()).values()) / max(G.number_of_nodes(), 1), 3),
    "relation_distribution": dict(relation_dist),
}

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\nKNOWLEDGE GRAPH METRICS")
for k, v in metrics.items():
    print(f"  {k}: {v}")



KNOWLEDGE GRAPH METRICS
  construction_method: LLM open-extraction (subject, relation, target) triples -> NetworkX DiGraph
  extraction_model: qwen2.5-8b-instruct-q4_k_m
  total_triples_processed: 76
  invalid_triples_skipped: 0
  unique_nodes: 38
  unique_edges: 43
  average_degree: 2.263
  relation_distribution: {'scans': 12, 'targets': 11, 'executes': 8, 'floods': 5, 'generates': 5, 'authenticates_to': 2}
